In [ ]:
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Sequence
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader    
import math
import json
from itertools import permutations

In [ ]:
import os, sys
sys.path.append('../../')

from pqcqec.utils.constants import QUBITS_FOR_GATES, QISKIT_GATES, GATE_IS_DIRECTIONAL


# Load Data

In [24]:
DATA_PATH = '../../data/json_data/3q_10g_5blk_data/'
GOOD_DATA_PATH = DATA_PATH + 'good_fidelity/'
BAD_DATA_PATH = DATA_PATH + 'poor_fidelity/'

good_data = []
poor_data = []

for filename in os.listdir(GOOD_DATA_PATH):
    with open(GOOD_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        good_data.append((token_dict['base_circuit_tokens'], token_dict['pqc_params'], token_dict['fidelity']))
        f.close()

for filename in os.listdir(BAD_DATA_PATH):
    with open(BAD_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        poor_data.append((token_dict['base_circuit_tokens'], token_dict['pqc_params'], token_dict['fidelity']))
        f.close()

print(f"Number of good data samples: {len(good_data)}")
print(f"Number of poor data samples: {len(poor_data)}")

Number of good data samples: 4270
Number of poor data samples: 731


In [25]:
good_data[0]

([['cx', [0, 1], []],
  ['h', [0], []],
  ['cx', [1, 0], []],
  ['cx', [1, 2], []],
  ['h', [1], []],
  ['x', [1], []],
  ['h', [0], []],
  ['cx', [2, 1], []],
  ['cx', [0, 2], []],
  ['h', [1], []],
  ['h', [1], []],
  ['cx', [0, 2], []],
  ['cx', [2, 1], []],
  ['h', [0], []],
  ['x', [1], []],
  ['h', [1], []],
  ['cx', [1, 2], []],
  ['cx', [1, 0], []],
  ['h', [0], []],
  ['cx', [0, 1], []]],
 [[[0.7436345815658569, 3.1218655109405518, 1.3743231296539307],
   [-2.506779909133911, 0.7037350535392761, 0.9809298515319824],
   [0.022151188924908638, 2.804871082305908, 0.027714822441339493]],
  [[-3.369378089904785, 1.6431126594543457, -1.8118082284927368],
   [1.874133586883545, 0.0831599235534668, -0.19617895781993866],
   [2.8753175735473633, 1.1417005062103271, -3.0622940063476562]],
  [[-0.7741491794586182, 0.9836245775222778, -2.3128905296325684],
   [-2.1487555503845215, -0.5787968635559082, 0.5645704865455627],
   [1.6096678972244263, -3.14855694770813, 1.8126161098480225]],
  

# Tokenizer for Data

In [ ]:
# ======================
# 1) Configuration
# ======================

@dataclass(frozen=True)
class TokenizerConfig:
    num_qubits: int                  # number of physical qubits in this dataset
    gates: Sequence[str]             # gate symbols expected in the data (e.g., ("H","X","CX"))
    max_len: int = 20                # #gate-steps per sequence (excludes CLS)
    add_cls: bool = True             # prepend a CLS position (useful for sequence pooling)
    make_pair_ids: bool = True       # emit control->target pair feature
    # NOTE: we assume "optional params" are absent/empty for now.


In [ ]:
# ======================
# 2) Tokenizer: ops -> integer features + masks
# ======================

class CircuitTokenizer:
    """
    Converts a list of ops (gate, qubits, params) into **aligned integer features**
    for a single sequence. Each op yields exactly one time step.

    Output (length L = max_len + add_cls):
      - gate_ids  : (L,) int      gate vocabulary ids (+ PAD, + CLS)
      - tgt_ids   : (L,) int      target qubit index (0..nq-1) or 0 when absent
      - c1_ids    : (L,) int      first control qubit or -1 when absent
      - c2_ids    : (L,) int      second control qubit or -1 when absent
      - pair_ids  : (L,) int      0 = "none", else 1 + ctrl*nq + tgt  (size nq^2 + 1)
      - attn_mask : (L,) int      1 for real tokens (incl. CLS), 0 for PAD
    """
    def __init__(self, cfg: TokenizerConfig):
        self.cfg = cfg

        # Build a compact gate vocabulary. We reserve **two extra ids**:
        #   PAD_ID: used for right-padding; CLS_ID: used for an optional CLS token.
        self.gate2id: Dict[str, int] = {g: i for i, g in enumerate(cfg.gates)}
        self.PAD_ID = len(self.gate2id)
        self.CLS_ID = self.PAD_ID + 1

        # Sanity: all gates must have metadata
        for g in self.gate2id:
            assert g in QUBITS_FOR_GATES, f"Missing arity for '{g}'."
            # assert g in GATE_IS_DIRECTIONAL,     f"Missing directionality for '{g}'."

    # ---- helpers (private) ----
    def _arity(self, g: str) -> int:
        return QUBITS_FOR_GATES[g]

    def _dir(self, g: str) -> bool:
        return GATE_IS_DIRECTIONAL.get(g, False)

    def _canon_qubits(self, g: str, qs: Sequence[int]) -> Tuple[int, ...]:
        """
        For symmetric gates we sort qubits (unique normal form).
        For directional gates we keep user-provided order.
        """
        return tuple(qs) if self._dir(g) else tuple(sorted(qs))

    def _assign_roles(self, g: str, qs: Tuple[int, ...]) -> Tuple[int, int, int]:
        """
        Map the qubit tuple to consistent roles: (target, ctrl1, ctrl2).

        k=1: (tgt=qs[0], c1=-1, c2=-1)
        k=2: directional -> (tgt=qs[1], c1=qs[0], c2=-1)
             symmetric   -> (tgt=qs[0], c1=qs[1], c2=-1)   (after sorting)
        k=3: directional -> (tgt=qs[2], c1=qs[0], c2=qs[1])
             symmetric   -> (tgt=qs[0], c1=qs[1], c2=qs[2])
        """
        k = len(qs)
        if k == 1:
            return qs[0], -1, -1
        if k == 2:
            return (qs[1], qs[0], -1) if self._dir(g) else (qs[0], qs[1], -1)
        if k == 3:
            return (qs[2], qs[0], qs[1]) if self._dir(g) else (qs[0], qs[1], qs[2])
        raise ValueError(f"Unsupported arity {k} for gate '{g}'.")

    def _pair_id(self, ctrl: int, tgt: int) -> int:
        """
        Encodes control->target as a small integer. We reserve **0 = 'none'** so
        actual pairs are 1..(nq^2). This avoids collisions with 'no pair'.
        """
        if ctrl < 0 or tgt < 0:
            return 0
        return 1 + ctrl * self.cfg.num_qubits + tgt

    # ---- main API ----
    def encode(
        self,
        circuit_ops: Sequence[Tuple[str, Sequence[int], Sequence[float]]],
        *,
        pad_to_max: bool = True,
    ) -> Dict[str, torch.Tensor]:
        """
        Convert one circuit (list of ops) to integer features + masks.
        """
        nq, max_len, add_cls = self.cfg.num_qubits, self.cfg.max_len, self.cfg.add_cls

        gate_ids: List[int] = []
        tgt_ids:  List[int] = []
        c1_ids:   List[int] = []
        c2_ids:   List[int] = []
        pair_ids: List[int] = []

        # (A) Optional CLS at position 0: it gets a **gate id** (CLS_ID)
        # and blank qubit roles; downstream we’ll mask roles on CLS so they don’t add noise.
        if add_cls:
            gate_ids.append(self.CLS_ID)
            tgt_ids.append(0)   # filler
            c1_ids.append(-1)   # absent
            c2_ids.append(-1)   # absent
            pair_ids.append(0)  # 'none'

        # (B) Encode each op as **one** time step
        for (gate, qubits, params) in circuit_ops:
            if gate not in self.gate2id:
                raise KeyError(f"Unknown gate '{gate}'. Allowed: {list(self.gate2id.keys())}")
            k = self._arity(gate)
            if len(qubits) != k:
                raise ValueError(f"Gate '{gate}' expects {k} qubits, got {len(qubits)}.")
            qs = tuple(int(q) for q in qubits)

            # Early validation: indices in-range and all distinct (e.g., ctrl != tgt)
            if any(q < 0 or q >= nq for q in qs):
                raise ValueError(f"Qubit out of range in {gate}{qs}; valid 0..{nq-1}.")
            if len(set(qs)) != len(qs):
                raise ValueError(f"Duplicate qubits in {gate}{qs} (e.g., ctrl==tgt).")

            # Canonicalize if symmetric (shrinks 'spelling' variants to one form)
            qs = self._canon_qubits(gate, qs)

            # Role assignment (gives the model a stable semantic layout)
            tgt, c1, c2 = self._assign_roles(gate, qs)

            # Append integer features
            gate_ids.append(self.gate2id[gate])
            tgt_ids.append(tgt)
            c1_ids.append(c1)
            c2_ids.append(c2)
            pair_ids.append(self._pair_id(c1, tgt) if self.cfg.make_pair_ids else 0)

        # (C) Right-pad / truncate to fixed length L
        L = max_len + (1 if add_cls else 0)

        def pad(xs: List[int], pad_val: int) -> List[int]:
            if len(xs) > L: return xs[:L]
            if len(xs) < L: return xs + [pad_val]*(L - len(xs))
            return xs

        gate_ids = pad(gate_ids, self.PAD_ID)
        tgt_ids  = pad(tgt_ids,  0)
        c1_ids   = pad(c1_ids,  -1)
        c2_ids   = pad(c2_ids,  -1)
        pair_ids = pad(pair_ids, 0)

        # (D) Attention mask: 1 for real tokens (incl. CLS), 0 for PAD
        real = min(len(circuit_ops) + (1 if add_cls else 0), L)
        attn_mask = [1]*real + [0]*(L - real)

        return {
            "gate_ids":  torch.tensor(gate_ids, dtype=torch.long),
            "tgt_ids":   torch.tensor(tgt_ids,  dtype=torch.long),
            "ctrl1_ids": torch.tensor(c1_ids,   dtype=torch.long),
            "ctrl2_ids": torch.tensor(c2_ids,   dtype=torch.long),
            "pair_ids":  torch.tensor(pair_ids, dtype=torch.long),
            "attn_mask": torch.tensor(attn_mask, dtype=torch.long),
        }

    @staticmethod
    def make_key_padding_mask(attn_mask: torch.Tensor) -> torch.Tensor:
        """
        PyTorch Transformer expects True where **PAD**.
        Input: attn_mask (B,L) with 1=real, 0=pad
        Output: key_padding_mask (B,L) with True=pad, False=keep
        """
        return (attn_mask == 0)

    def __repr__(self) -> str:
        return (f"CircuitTokenizer(nq={self.cfg.num_qubits}, gates={list(self.gate2id.keys())}, "
                f"max_len={self.cfg.max_len}, add_cls={self.cfg.add_cls}, "
                f"vocab={len(self.gate2id)+2})")  # +PAD +CLS


In [ ]:
# ======================
# 3) Encoder: integer features -> token embeddings (B,L,D)
# ======================

class CircuitEncoder(nn.Module):
    """
    Turns the tokenizer's integer features into dense token vectors by summing
    role-specific embeddings. These vectors go straight into a Transformer.

    Embeddings:
      - E_gate:  gate type (+PAD +CLS)
      - E_tgt :  target qubit
      - E_c1  :  control-1 qubit
      - E_c2  :  control-2 qubit (only used by 3-qubit gates)
      - E_pair:  control->target small table (size nq^2 + 1, with 0='none')

    All role embeddings are **masked** on PAD positions, and also masked on CLS
    (so CLS doesn’t accidentally pick up a qubit role vector).
    """
    def __init__(self, gate_vocab_size: int, num_qubits: int, d_model: int,
                 use_pair: bool = True, dropout: float = 0.1):
        super().__init__()
        self.nq = num_qubits
        self.use_pair = use_pair

        # +2 to include PAD and CLS ids used by the tokenizer
        self.E_gate = nn.Embedding(gate_vocab_size + 2, d_model)

        # Separate role tables let the model learn different semantics per role
        self.E_tgt  = nn.Embedding(num_qubits, d_model)
        self.E_c1   = nn.Embedding(num_qubits, d_model)
        self.E_c2   = nn.Embedding(num_qubits, d_model)

        if use_pair:
            # +1 because tokenizer reserves pair_id==0 for 'none'
            self.E_pair = nn.Embedding(num_qubits * num_qubits + 1, d_model)

        self.dropout = nn.Dropout(dropout)

        # Small learned scalars to balance contributions (optional but handy)
        self.alpha_gate = nn.Parameter(torch.tensor(1.0))
        self.alpha_role = nn.Parameter(torch.tensor(1.0))
        self.alpha_pair = nn.Parameter(torch.tensor(1.0))

    def forward(self, batch_feats: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Inputs (all (B,L)):
          batch_feats["gate_ids"], ["tgt_ids"], ["ctrl1_ids"], ["ctrl2_ids"],
          ["pair_ids"], ["attn_mask"]

        Returns:
          x : (B,L,D) token embeddings for the Transformer
          key_padding_mask : (B,L) bool, True where PAD
        """
        gate_ids  = batch_feats["gate_ids"]   # (B,L)
        tgt_ids   = batch_feats["tgt_ids"]
        c1_ids    = batch_feats["ctrl1_ids"]
        c2_ids    = batch_feats["ctrl2_ids"]
        pair_ids  = batch_feats["pair_ids"]
        attn_mask = batch_feats["attn_mask"]  # 1=real (incl. CLS), 0=PAD

        B, L = gate_ids.shape
        device = gate_ids.device

        # (1) Which positions are padding? Which positions are CLS?
        is_pad = (attn_mask == 0)                         # (B,L) bool
        # CLS id is tokenizer-dependent: last id (= PAD_ID + 1).
        CLS_ID = self.E_gate.num_embeddings - 1
        is_cls = (gate_ids == CLS_ID)                     # (B,L) bool

        # (2) Base: gate type embedding (always applied on non-PAD positions)
        x = self.E_gate(gate_ids) * self.alpha_gate       # (B,L,D)

        # (3) Add role embeddings, but **mask** PAD and CLS (CLS has no roles)
        not_pad_nor_cls = (~is_pad) & (~is_cls)           # (B,L)
        # Build masks for controls: -1 means 'absent' ⇒ mask those positions
        has_c1 = (c1_ids >= 0) & not_pad_nor_cls
        has_c2 = (c2_ids >= 0) & not_pad_nor_cls

        # Safe indices (clamped) where present; elsewhere values won’t be used (masked)
        c1_safe = c1_ids.clamp(min=0)
        c2_safe = c2_ids.clamp(min=0)

        # Target (always present for non-PAD, non-CLS)
        x = x + self.alpha_role * (self.E_tgt(tgt_ids) * not_pad_nor_cls.unsqueeze(-1))
        # Controls (only where present)
        x = x + self.alpha_role * (self.E_c1(c1_safe) * has_c1.unsqueeze(-1))
        x = x + self.alpha_role * (self.E_c2(c2_safe) * has_c2.unsqueeze(-1))

        # (4) Optional: pair embedding for control->target (0 = 'none')
        if self.use_pair:
            x = x + self.alpha_pair * (self.E_pair(pair_ids) * not_pad_nor_cls.unsqueeze(-1))

        # (5) Final dropout before Transformer
        x = self.dropout(x)

        # (6) Build key_padding_mask expected by nn.TransformerEncoder
        key_padding_mask = is_pad  # True where PAD

        return x, key_padding_mask

In [27]:
tokenizer = CircuitTokenizer(num_qubits=3, gates=['x', 'h', 'cx'])
tokenizer.tokens_dict

Total tokens: 15


{('<UNK>', ()): 0,
 ('<PAD>', ()): 1,
 ('<CLS>', ()): 2,
 ('x', (0,)): 3,
 ('x', (1,)): 4,
 ('x', (2,)): 5,
 ('h', (0,)): 6,
 ('h', (1,)): 7,
 ('h', (2,)): 8,
 ('cx', (0, 1)): 9,
 ('cx', (0, 2)): 10,
 ('cx', (1, 0)): 11,
 ('cx', (1, 2)): 12,
 ('cx', (2, 0)): 13,
 ('cx', (2, 1)): 14}

In [28]:
good_data[0][0]

[['cx', [0, 1], []],
 ['h', [0], []],
 ['cx', [1, 0], []],
 ['cx', [1, 2], []],
 ['h', [1], []],
 ['x', [1], []],
 ['h', [0], []],
 ['cx', [2, 1], []],
 ['cx', [0, 2], []],
 ['h', [1], []],
 ['h', [1], []],
 ['cx', [0, 2], []],
 ['cx', [2, 1], []],
 ['h', [0], []],
 ['x', [1], []],
 ['h', [1], []],
 ['cx', [1, 2], []],
 ['cx', [1, 0], []],
 ['h', [0], []],
 ['cx', [0, 1], []]]

In [29]:
tokenizer.encode(good_data[0][0])

[9, 6, 11, 12, 7, 4, 6, 14, 10, 7, 7, 10, 14, 6, 4, 7, 12, 11, 6, 9]